In [ ]:
!pip install requests pandas tqdm -q

In [ ]:
from google.colab import files
print("Upload trains.csv AND train_schedule.csv (needed for station code lookup)")
uploaded = files.upload()

Upload trains.csv AND train_schedule.csv (needed for station code lookup)


Saving train_schedule.csv to train_schedule.csv
Saving trains.csv to trains.csv


In [ ]:
import pandas as pd
import requests
import re
import time
import os
import csv
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

TRAINS_CSV = "trains.csv"
SCHEDULE_CSV = "train_schedule.csv"
OUT_TRAINS = "running_days_scraped.csv"
OUT_SCHEDULE = "train_schedule_scraped.csv"
SLEEP_SECONDS = 0.1
TIMEOUT = 15
MAX_RETRIES = 2
LIMIT = None   # start VERY small - this scrapes much more per train than before

DAY_MAP = {"M": 0, "Tu": 1, "W": 2, "Th": 3, "F": 4, "Sa": 5, "Su": 6}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

ROUTE_ROW_PATTERN = re.compile(
    r'(\d+)\s+([A-Za-z][A-Za-z .()\-]*?)\s+(\d{2}\.\d{2}|First)\s+(\d{2}\.\d{2}|Last)\s+(\d+)\s+([\d,]*?)(?=\s*\d+\s+[A-Z]|\s*$)'
)
SUMMARY_PATTERN_TEMPLATE = r'Abount Train\s+{num}[^,]*,\s*(.+?)\s+to\s+(.+?)\s+runs\s+([A-Za-z ,]+?),\s*has classes'

def time_to_minutes(t):
    if t in ("First", "Last"):
        return None
    h, m = t.split(".")
    return int(h) * 60 + int(m)

def build_station_code_lookup(schedule_df):
    """Map normalized station_name -> station_code, from your existing schedule file."""
    pairs = schedule_df[["station_name", "station_code"]].dropna().drop_duplicates()
    lookup = {}
    for _, row in pairs.iterrows():
        key = re.sub(r'[^a-z]', '', str(row["station_name"]).lower())
        lookup[key] = row["station_code"]
    return lookup

def lookup_station_code(name, code_lookup):
    key = re.sub(r'\b(jn|jct|junction)\b', '', name.lower())
    key = re.sub(r'[^a-z]', '', key)
    return code_lookup.get(key, "")

def parse_summary(text, train_number):
    pattern = SUMMARY_PATTERN_TEMPLATE.format(num=re.escape(train_number))
    match = re.search(pattern, text)
    if not match:
        return None
    source, destination, schedule_text = match.groups()
    schedule_text = schedule_text.strip()
    if schedule_text.lower() == "daily":
        running_days = "1111111"
    else:
        bits = ["0"] * 7
        tokens = schedule_text.split()
        if any(t not in DAY_MAP for t in tokens):
            return None
        for t in tokens:
            bits[DAY_MAP[t]] = "1"
        running_days = "".join(bits)
    return {
    "train_name": train_name.strip(),
    "source": source.strip(),
    "destination": destination.strip(),
    "running_days": running_days,
}

def parse_train_name(text, train_number):
    m = re.search(
        rf'Abount Train\s+{re.escape(train_number)}\s+(.+?),',
        text,
        re.IGNORECASE
    )

    if not m:
        return ""

    train_name = m.group(1).strip()

    # Remove repeated train number and anything after it
    train_name = re.split(rf'\s+{re.escape(train_number)}\b', train_name)[0].strip()

    return train_name

def parse_route(text):
    rows = ROUTE_ROW_PATTERN.findall(text)
    stops = []
    day = 1
    prev_minutes = None
    for seq, name, arr, dep, dist, platform in rows:
        arr_min = time_to_minutes(arr)
        dep_min = time_to_minutes(dep)
        check_time = arr_min if arr_min is not None else dep_min
        if prev_minutes is not None and check_time is not None and check_time < prev_minutes:
            day += 1
        stops.append({
            "seq": int(seq), "station_name": name.strip(),
            "arrival": "00:00:00" if arr == "First" else arr.replace(".", ":") + ":00",
            "departure": "00:00:00" if dep == "Last" else dep.replace(".", ":") + ":00",
            "day": day,
        })
        prev_minutes = dep_min if dep_min is not None else arr_min
    return stops

def normalize_station(name):
    name = str(name).strip().lower()
    name = re.sub(r'\b(jn|jct|junction|terminus|term|central|cantt|cant)\b', '', name)
    name = re.sub(r'[^a-z]', '', name)
    return name

def station_match(a, b):
    if not a or not b:
        return "unknown"
    a, b = normalize_station(a), normalize_station(b)
    if not a or not b:
        return "unknown"
    return "match" if (a in b or b in a) else "MISMATCH"

def combined_check(src, dst):
    if "match" in (src, dst):
        return "match"
    if src == "unknown" and dst == "unknown":
        return "unknown"
    return "MISMATCH"

def fetch_train_page(train_number):
    url = f"https://erail.in/train-enquiry/{train_number}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code == 404:
                return None, "not_found"
            resp.raise_for_status()
            return resp.text, "ok"
        except Exception:
            time.sleep(2 * attempt)
    return None, "fetch_failed"

trains_df = pd.read_csv(TRAINS_CSV, dtype=str).dropna(subset=["train_number"])
trains_df = trains_df.drop_duplicates(subset=["train_number"])
schedule_df = pd.read_csv(SCHEDULE_CSV, dtype=str)
code_lookup = build_station_code_lookup(schedule_df)

before = len(trains_df)
trains_df = trains_df[~trains_df["train_name"].str.contains("sp", case=False, na=False)]
print(f"Total: {before} | Skipped (special): {before - len(trains_df)} | To process: {len(trains_df)}")

if LIMIT is not None:
    trains_df = trains_df.head(LIMIT)

done_numbers = set()
if os.path.exists(OUT_TRAINS):
    done_numbers = set(pd.read_csv(OUT_TRAINS, dtype=str)["train_number"])
to_fetch = trains_df[~trains_df["train_number"].isin(done_numbers)]
print(f"Already done: {len(done_numbers)} | Remaining: {len(to_fetch)}")

trains_file_exists = os.path.exists(OUT_TRAINS)
schedule_file_exists = os.path.exists(OUT_SCHEDULE)

with open(OUT_TRAINS, "a", newline="", encoding="utf-8") as tf, \
     open(OUT_SCHEDULE, "a", newline="", encoding="utf-8") as sf:

    t_writer = csv.writer(tf)
    s_writer = csv.writer(sf)
    if not trains_file_exists:
        t_writer.writerow([
    "train_number",
    "train_name",
    "scraped_train_name",
    "csv_source",
    "csv_destination",
    "scraped_source",
    "scraped_destination",
    "identity_check",
    "running_days",
    "status"
])
    if not schedule_file_exists:
        s_writer.writerow([
    "train_number",
    "train_name",
    "scraped_train_name",
    "day",
    "station_name",
    "station_code",
    "arrival",
    "departure"
])

    for row in tqdm(to_fetch.itertuples(), total=len(to_fetch), desc="Scraping", unit="train"):
        train_number = row.train_number
        train_name = getattr(row, "train_name", "")
        csv_source = getattr(row, "source", "")
        csv_destination = getattr(row, "destination", "")

        html, fetch_status = fetch_train_page(train_number)
        if html is None:
            t_writer.writerow([
    train_number,
    train_name,
    "",
    csv_source,
    csv_destination,
    "",
    "",
    "unknown",
    "1111111",
    fetch_status
])
        else:
            soup = BeautifulSoup(html, "html.parser")
            text = re.sub(r'\s+', ' ', soup.get_text(separator=" "))
            scraped_train_name = parse_train_name(text, train_number)

            summary = parse_summary(text, train_number)
            stops = parse_route(text)

            if summary is None:
                t_writer.writerow([
    train_number,
    train_name,
    scraped_train_name,
    csv_source,
    csv_destination,
    "",
    "",
    "unknown",
    "1111111",
    "summary_parse_failed"
])
            else:
                src_check = station_match(csv_source, summary["source"])
                dst_check = station_match(csv_destination, summary["destination"])
                id_check = combined_check(src_check, dst_check)
                t_writer.writerow([
    train_number,
    train_name,
    scraped_train_name,
    csv_source,
    csv_destination,
    summary["source"],
    summary["destination"],
    id_check,
    summary["running_days"],
    "ok"
])

            for stop in stops:
                code = lookup_station_code(stop["station_name"], code_lookup)
                s_writer.writerow([
    train_number,
    train_name,
    scraped_train_name,
    stop["day"],
    stop["station_name"],
    code,
    stop["arrival"],
    stop["departure"]
])

        tf.flush()
        sf.flush()
        time.sleep(SLEEP_SECONDS)

print("Done!")
print(f"Train-level file: {OUT_TRAINS}")
print(f"Route/schedule file: {OUT_SCHEDULE}")

Total: 5207 | Skipped (special): 209 | To process: 4998
Already done: 10 | Remaining: 4988


Scraping:   0%|          | 0/4988 [00:00<?, ?train/s]

Done!
Train-level file: running_days_scraped.csv
Route/schedule file: train_schedule_scraped.csv


#code2

In [ ]:
import pandas as pd
import requests
import re
import time
import os
import csv
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

TRAINS_CSV = "trains.csv"
SCHEDULE_CSV = "train_schedule.csv"
OUT_TRAINS = "running_days_scraped.csv"
OUT_SCHEDULE = "train_schedule_scraped.csv"
SLEEP_SECONDS = 1
TIMEOUT = 15
MAX_RETRIES = 2
LIMIT = 5   # start VERY small - this scrapes much more per train than before

DAY_MAP = {"M": 0, "Tu": 1, "W": 2, "Th": 3, "F": 4, "Sa": 5, "Su": 6}
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

ROUTE_ROW_PATTERN = re.compile(
    r'(\d+)\s+([A-Za-z][A-Za-z .()\-]*?)\s+(\d{2}\.\d{2}|First)\s+(\d{2}\.\d{2}|Last)\s+(\d+)\s+([\d,]*?)(?=\s*\d+\s+[A-Z]|\s*$)'
)
SUMMARY_PATTERN_TEMPLATE = (
    r'Abount Train\s+{num}\s+(.+?),\s*'
    r'(.+?)\s+to\s+(.+?)\s+runs\s+([A-Za-z ,]+?),\s*has classes'
)

def time_to_minutes(t):
    if t in ("First", "Last"):
        return None
    h, m = t.split(".")
    return int(h) * 60 + int(m)

def build_station_code_lookup(schedule_df):
    """Map normalized station_name -> station_code, from your existing schedule file."""
    pairs = schedule_df[["station_name", "station_code"]].dropna().drop_duplicates()
    lookup = {}
    for _, row in pairs.iterrows():
        key = re.sub(r'[^a-z]', '', str(row["station_name"]).lower())
        lookup[key] = row["station_code"]
    return lookup

def lookup_station_code(name, code_lookup):
    key = re.sub(r'\b(jn|jct|junction)\b', '', name.lower())
    key = re.sub(r'[^a-z]', '', key)
    return code_lookup.get(key, "")

def parse_summary(text, train_number):
    pattern = SUMMARY_PATTERN_TEMPLATE.format(num=re.escape(train_number))
    match = re.search(pattern, text)
    if not match:
        return None
    source, destination, schedule_text = match.groups()
    schedule_text = schedule_text.strip()
    if schedule_text.lower() == "daily":
        running_days = "1111111"
    else:
        bits = ["0"] * 7
        tokens = schedule_text.split()
        if any(t not in DAY_MAP for t in tokens):
            return None
        for t in tokens:
            bits[DAY_MAP[t]] = "1"
        running_days = "".join(bits)
    return {"source": source.strip(), "destination": destination.strip(), "running_days": running_days}

def parse_route(text):
    rows = ROUTE_ROW_PATTERN.findall(text)
    stops = []
    day = 1
    prev_minutes = None
    for seq, name, arr, dep, dist, platform in rows:
        arr_min = time_to_minutes(arr)
        dep_min = time_to_minutes(dep)
        check_time = arr_min if arr_min is not None else dep_min
        if prev_minutes is not None and check_time is not None and check_time < prev_minutes:
            day += 1
        stops.append({
            "seq": int(seq), "station_name": name.strip(),
            "arrival": "00:00:00" if arr == "First" else arr.replace(".", ":") + ":00",
            "departure": "00:00:00" if dep == "Last" else dep.replace(".", ":") + ":00",
            "day": day,
        })
        prev_minutes = dep_min if dep_min is not None else arr_min
    return stops

def normalize_station(name):
    name = str(name).strip().lower()
    name = re.sub(r'\b(jn|jct|junction|terminus|term|central|cantt|cant)\b', '', name)
    name = re.sub(r'[^a-z]', '', name)
    return name

def station_match(a, b):
    if not a or not b:
        return "unknown"
    a, b = normalize_station(a), normalize_station(b)
    if not a or not b:
        return "unknown"
    return "match" if (a in b or b in a) else "MISMATCH"

def combined_check(src, dst):
    if "match" in (src, dst):
        return "match"
    if src == "unknown" and dst == "unknown":
        return "unknown"
    return "MISMATCH"

def fetch_train_page(train_number):
    url = f"https://erail.in/train-enquiry/{train_number}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code == 404:
                return None, "not_found"
            resp.raise_for_status()
            return resp.text, "ok"
        except Exception:
            time.sleep(2 * attempt)
    return None, "fetch_failed"

trains_df = pd.read_csv(TRAINS_CSV, dtype=str).dropna(subset=["train_number"])
trains_df = trains_df.drop_duplicates(subset=["train_number"])
schedule_df = pd.read_csv(SCHEDULE_CSV, dtype=str)
code_lookup = build_station_code_lookup(schedule_df)

before = len(trains_df)
trains_df = trains_df[~trains_df["train_name"].str.contains("special", case=False, na=False)]
print(f"Total: {before} | Skipped (special): {before - len(trains_df)} | To process: {len(trains_df)}")

if LIMIT is not None:
    trains_df = trains_df.head(LIMIT)

done_numbers = set()
if os.path.exists(OUT_TRAINS):
    done_numbers = set(pd.read_csv(OUT_TRAINS, dtype=str)["train_number"])
to_fetch = trains_df[~trains_df["train_number"].isin(done_numbers)]
print(f"Already done: {len(done_numbers)} | Remaining: {len(to_fetch)}")

trains_file_exists = os.path.exists(OUT_TRAINS)
schedule_file_exists = os.path.exists(OUT_SCHEDULE)

with open(OUT_TRAINS, "a", newline="", encoding="utf-8") as tf, \
     open(OUT_SCHEDULE, "a", newline="", encoding="utf-8") as sf:

    t_writer = csv.writer(tf)
    s_writer = csv.writer(sf)
    if not trains_file_exists:
        t_writer.writerow(["train_number", "train_name", "csv_source", "csv_destination",
                            "scraped_source", "scraped_destination", "identity_check",
                            "running_days", "status"])
    if not schedule_file_exists:
        s_writer.writerow(["train_number", "train_name", "day", "station_name",
                            "station_code", "arrival", "departure"])

    for row in tqdm(to_fetch.itertuples(), total=len(to_fetch), desc="Scraping", unit="train"):
        train_number = row.train_number
        train_name = getattr(row, "train_name", "")
        csv_source = getattr(row, "source", "")
        csv_destination = getattr(row, "destination", "")

        html, fetch_status = fetch_train_page(train_number)
        if html is None:
            t_writer.writerow([train_number, train_name, csv_source, csv_destination,
                                "", "", "unknown", "1111111", fetch_status])
        else:
            soup = BeautifulSoup(html, "html.parser")
            text = re.sub(r'\s+', ' ', soup.get_text(separator=" "))

            summary = parse_summary(text, train_number)
            stops = parse_route(text)

            if summary is None:
                t_writer.writerow([train_number, train_name, csv_source, csv_destination,
                                    "", "", "unknown", "1111111", "summary_parse_failed"])
            else:
                src_check = station_match(csv_source, summary["source"])
                dst_check = station_match(csv_destination, summary["destination"])
                id_check = combined_check(src_check, dst_check)
                t_writer.writerow([train_number, train_name, csv_source, csv_destination,
                                    summary["source"], summary["destination"],
                                    id_check, summary["running_days"], "ok"])

            for stop in stops:
                code = lookup_station_code(stop["station_name"], code_lookup)
                s_writer.writerow([train_number, train_name, stop["day"], stop["station_name"],
                                    code, stop["arrival"], stop["departure"]])

        tf.flush()
        sf.flush()
        time.sleep(SLEEP_SECONDS)

print("Done!")
print(f"Train-level file: {OUT_TRAINS}")
print(f"Route/schedule file: {OUT_SCHEDULE}")